# First ResNet

imports

In [ ]:
import os
import torch
import torch.nn as nn
import librosa 
import random
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, Subset

In [ ]:
#class PitchShift:
    #def __init__(self, n_steps_range=(-2, 2), sr=22050):
        self.n_steps_range = n_steps_range  # Pitch shift range (in semitones)
        self.sr = sr  # Sample rate

   # def __call__(self, y):
        # Randomly select the pitch shift amount within the specified range
        n_steps = random.uniform(*self.n_steps_range)
        return librosa.effects.pitch_shift(y, sr=self.sr, n_steps=n_steps)


In [2]:
# ✅ Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# ✅ Step 1: Define Data Transforms (No resizing required)
data_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # Convert grayscale to RGB for ResNet
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],  # Mean normalization for pretrained ResNet
                         [0.229, 0.224, 0.225])  # Standard deviation normalization
])

In [4]:
# ✅ Step 2: Load Dataset from 224x224 spectrogram folder
data_dir = "test_batch"  # Root directory with spectrograms
dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)

# Split dataset into train (80%) and validation (20%)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [6]:
# ✅ Step 3: Load Pretrained ResNet
model = models.resnet50(pretrained=True)

# Modify the final layer for 3 output classes (LC, VN, EN)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 3)

# Send model to device
model = model.to(device)

/Users/tatlowdd/Desktop/Ornistone/toy_model/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/tatlowdd/Desktop/Ornistone/toy_model/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# ✅ Step 4: Freeze earlier layers (optional)
for param in model.parameters():
    param.requires_grad = False  # Freeze all layers

# Unfreeze some layers to see how it affects things
for param in model.layer4.parameters():
    param.requires_grad = True

# Unfreeze final fully connected layer
for param in model.fc.parameters():
    param.requires_grad = True

In [8]:
# ✅ Step 5: Define Loss Function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.0005)

In [9]:
# ✅ Step 6: Training Loop
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    epoch_loss = running_loss / len(train_loader.dataset)

    # ✅ Validation Step
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

# ✅ Step 7: Save the Trained Model
torch.save(model.state_dict(), "resnet_bird_224x224_round3.pth")

Epoch 1/30, Loss: 1.0156, Train Acc: 0.5375, Val Acc: 0.5888
Epoch 2/30, Loss: 0.9370, Train Acc: 0.5676, Val Acc: 0.5668
Epoch 3/30, Loss: 0.8955, Train Acc: 0.5900, Val Acc: 0.6193
Epoch 4/30, Loss: 0.8789, Train Acc: 0.6044, Val Acc: 0.6447
Epoch 5/30, Loss: 0.8562, Train Acc: 0.6201, Val Acc: 0.6345
Epoch 6/30, Loss: 0.8438, Train Acc: 0.6264, Val Acc: 0.5956
Epoch 7/30, Loss: 0.8295, Train Acc: 0.6387, Val Acc: 0.6362
Epoch 8/30, Loss: 0.8260, Train Acc: 0.6269, Val Acc: 0.6464
Epoch 9/30, Loss: 0.8106, Train Acc: 0.6417, Val Acc: 0.6548
Epoch 10/30, Loss: 0.8061, Train Acc: 0.6391, Val Acc: 0.5905
Epoch 11/30, Loss: 0.7987, Train Acc: 0.6442, Val Acc: 0.6531
Epoch 12/30, Loss: 0.7938, Train Acc: 0.6612, Val Acc: 0.6565
Epoch 13/30, Loss: 0.7715, Train Acc: 0.6739, Val Acc: 0.6108
Epoch 14/30, Loss: 0.7792, Train Acc: 0.6459, Val Acc: 0.6599
Epoch 15/30, Loss: 0.7748, Train Acc: 0.6650, Val Acc: 0.6565
Epoch 16/30, Loss: 0.7695, Train Acc: 0.6540, Val Acc: 0.6684
Epoch 17/30, Loss